# Optional A: How to build the same pipeline with LlamaIndex

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | Self-study**

## Goal

Rebuild the pipeline of Labs 01, 02 and 05 with LlamaIndex, a framework made for RAG, and check it question by question
against your own code. The moment that teaches it: both pipelines find the right paragraph for every question, yet
they return the same three chunks in the same order for only about half the questions. LlamaIndex embeds each chunk's metadata with its
text unless you tell it not to, and the overall score could not show it.

## What you will learn

- How to turn chunk records into LlamaIndex `TextNode`s that keep their IDs, and print the text the embedding model and
  the LLM read from each node.
- How to build LlamaIndex's chat model and embeddings from `models.env` with the `openai-like` integrations, and what
  LlamaIndex uses when you set nothing.
- How to index nodes in a `VectorStoreIndex`, search them with a retriever, and find its default number of results.
- How to check a framework's search against your own, question by question, and find the setting that made them differ.
- How to read a query engine's prompt, and get a cited answer with `output_cls` that Lab 02's quote check can check.

## Set up the notebook

This lab runs in its own environment, not in the repository's `.venv`: LlamaIndex's OpenAI integration needs `openai`
below version 3, and the other labs use version 3. Create it as the README shows and choose its kernel
(`Optional_LlamaIndex_Pipeline/.venv`). **Setup** finds the repository folder (on Colab it copies the repository and
installs this lab's packages first) and prints the two versions that matter. **Config** reads the settings and finds
your key.

In [1]:
# Setup: on Colab, copy the repository and install this lab's packages. Elsewhere, use this lab's own environment (README).
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r Optional_LlamaIndex_Pipeline/requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

import openai
import llama_index.core
print("openai", openai.__version__, "| llama-index-core", llama_index.core.__version__)

Repository folder: rag_for_nlp


openai 2.54.0 | llama-index-core 0.14.25


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above. It embeds the corpus four times (about 21,000 tokens
each) and makes about 70 short chat calls, well under one US cent on `gpt-4o-mini`. The saved outputs and every count in
the text come from `gpt-4o-mini`; on another model the failures can move to other questions. Model replies vary and scores can move
in the third decimal, so compare the order of results and the kind of reply, not every number.

[LlamaIndex](https://developers.llamaindex.ai/python/framework/) is a framework built around RAG: it cuts documents into
**nodes**, indexes them and answers questions over them, with many defaults chosen for you. As in Lab 05, you check it
against your own pipeline before you trust it. *Plain* is your pipeline from Labs 01 and 02 (`shared/search.py` and
`shared/answer.py`); this environment has no LangChain, and Lab 05 showed that LangChain finds the same passages as plain Python once its defaults are set.

---

## Part A: Turn the chunks into nodes

Part A gives LlamaIndex the corpus in the shape its index takes.

Load the same 277 records as Lab 05, with `corpus()` from `shared/trace.py` (the 272 chunks and the five policies), and
the 30 development questions:

In [3]:
import json
from shared.trace import corpus as load_corpus

corpus = load_corpus()
questions = json.loads((ROOT / "data" / "development-questions.json").read_text(encoding="utf-8"))
print(len(corpus), "records |", len(questions), "questions")
print("a chunk: ", list(corpus[0]))
print("a policy:", list(corpus[-1]))

277 records | 30 questions
a chunk:  ['chunk_id', 'source_id', 'title', 'start', 'end', 'text', 'tokens']
a policy: ['source_id', 'title', 'text', 'version', 'access', 'chunk_id']


LlamaIndex has two text objects. A **`Document`** is a whole source, such as a file; when you index documents,
LlamaIndex cuts each one into **nodes** with its own splitter. A **`TextNode`** is one piece that is already cut, with
an ID, its text and a `metadata` dictionary
([documents and nodes](https://developers.llamaindex.ai/python/framework/module_guides/loading/documents_and_nodes/usage_documents/)).
Our chunks are already cut, and their IDs are what a citation names, so build nodes directly. One by hand, from the
first record:

In [4]:
from llama_index.core.schema import TextNode

record = corpus[0]
node = TextNode(id_=record["chunk_id"], text=record["text"],
                metadata={name: value for name, value in record.items() if name != "text"})
print("id:      ", node.node_id)
print("metadata:", node.metadata)
print("text:    ", node.text[:80], "...")

id:       S00-P00-C00
metadata: {'chunk_id': 'S00-P00-C00', 'source_id': 'S00-P00', 'title': 'Super_Bowl_50', 'start': 0, 'end': 486, 'tokens': 98}
text:     Super Bowl 50 was an American football game to determine the champion of the Nat ...


The same layout as Lab 05's `Document`: the text, the chunk ID as the node's ID, and every other field in `metadata`,
where `source_id` is kept for hit@3. Now all 277:

In [5]:
nodes = [TextNode(id_=record["chunk_id"], text=record["text"],
                  metadata={name: value for name, value in record.items() if name != "text"})
         for record in corpus]
print(len(nodes), "nodes | last:", nodes[-1].node_id, nodes[-1].metadata)

277 nodes | last: H05 {'source_id': 'H05', 'title': 'Atlas billing', 'version': '2026-10-01', 'access': 'public', 'chunk_id': 'H05'}


### Checkpoint A

| You should see | Our run |
|---|---|
| Corpus | 277 records and 30 questions; chunks with `start`, `end` and `tokens`, policies with `version` and `access` |
| Nodes | 277; the last is H05, with its title, `version`, `access` and `chunk_id` in the metadata |

---

## Part B: Build the models from the settings

Part B gives LlamaIndex the embedding model and chat model that `models.env` names, and shows what it would use
otherwise.

LlamaIndex keeps its defaults in one object, **`Settings`**
([Settings](https://developers.llamaindex.ai/python/framework/module_guides/supporting_modules/settings/)): every index
and query engine that is not given a model takes it from there. Ask it what it holds before you set anything:

In [6]:
from llama_index.core import Settings

print("Settings.llm:        ", type(Settings.llm).__name__, Settings.llm.model)
print("Settings.embed_model:", type(Settings.embed_model).__name__, Settings.embed_model.model_name)
print("Settings.chunk_size: ", Settings.chunk_size)

Settings.llm:         OpenAI gpt-3.5-turbo
Settings.embed_model: OpenAIEmbedding text-embedding-ada-002
Settings.chunk_size:  1024


Nothing was called, but LlamaIndex built OpenAI models of its own choice, `gpt-3.5-turbo` and `text-embedding-ada-002`, neither
of them in `models.env`. An index built now would embed your corpus with another model than Lab 01 used, without a
warning. `chunk_size` is what its splitter cuts documents to; our nodes are cut already.

⚠️ Print the fields you need, never a whole model object: LlamaIndex's model objects print every field, your API key
included.

The **`openai-like`** integrations are LlamaIndex's classes for any OpenAI-compatible server: you give them the model,
the address and the key
([`OpenAILikeEmbedding`](https://developers.llamaindex.ai/python/framework-api-reference/embeddings/openai_like/)).
Build the embedding model from the settings and compare one vector with Lab 01's `embed()`:

In [7]:
import os
import numpy as np
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from shared.settings import embed_url
from shared.search import embed

embed_model = OpenAILikeEmbedding(model_name=settings["EMBED_MODEL"], api_base=embed_url(settings),
                                  api_key=os.environ["OPENAI_API_KEY"])
question = "In what year was Nikola Tesla born?"
li_vector = np.array(embed_model.get_query_embedding(question))
plain_vector = embed([question], settings)[0]
print("numbers:", len(li_vector), "| largest difference from embed():", f"{np.abs(li_vector - plain_vector).max():.6f}")
print("texts per request:", embed_model.embed_batch_size)

numbers: 1536 | largest difference from embed(): 0.000000
texts per request: 10


The same model gives the same vector: 1,536 numbers, largest difference 0.000000. LlamaIndex sends
10 texts per request where `embed()` sends 100, so indexing the corpus takes 28 requests instead of 3; slower,
same vectors. If your `EMBED_BASE_URL` points to another provider, pass `EMBED_API_KEY` as the key instead.

Now the chat model, [`OpenAILike`](https://developers.llamaindex.ai/python/framework-api-reference/llms/openai_like/).
`chat_options()` from Lab 02 gives what every chat call adds (the model, and `temperature` or `reasoning_effort`);
`additional_kwargs` adds them to every request. Build it with only those, and ask it a question:

In [8]:
from llama_index.llms.openai_like import OpenAILike
from shared.settings import chat_url, chat_options

options = chat_options("CHAT_MODEL", settings)
first_llm = OpenAILike(model=settings["CHAT_MODEL"], api_base=chat_url(settings), api_key=os.environ["OPENAI_API_KEY"],
                       additional_kwargs=options)
print("is_chat_model:", first_llm.is_chat_model, "| context_window:", first_llm.context_window)
try:
    print(repr(first_llm.complete("In one sentence, what does a vector store do?").text))
except Exception as error:      # some models have no Completions route; see Troubleshooting
    print(f"{type(error).__name__}: {str(error)[:100]}")

is_chat_model: False | context_window: 3900


'**\nA vector store is a database designed to efficiently store, retrieve, and manage'


The reply starts with `**` and a line break, as if your question were the start of a longer text, and stops in the middle of a sentence. `OpenAILike` starts with `is_chat_model=False`, so it sent your text as a prompt to
continue to the older Completions endpoint, `/v1/completions`, not to Chat Completions, and that endpoint stops after
16 tokens unless you ask for more. Some servers and models do not offer it at all. Set `is_chat_model=True` and ask
again:

In [9]:
llm = OpenAILike(model=settings["CHAT_MODEL"], api_base=chat_url(settings), api_key=os.environ["OPENAI_API_KEY"],
                 is_chat_model=True, additional_kwargs=options)
print(repr(llm.complete("In one sentence, what does a vector store do?").text))
print("added to every request:", llm.additional_kwargs)

'A vector store is a database designed to efficiently store, retrieve, and manage high-dimensional vectors, often used for tasks like similarity search and machine learning applications.'
added to every request: {'model': 'gpt-4o-mini', 'temperature': 0}


A full answer, from Chat Completions. `additional_kwargs` holds what `chat_options()` gave and wins over the object's
own `temperature` field. Put both models in `Settings`, so every index and engine in this notebook uses them:

In [10]:
Settings.llm = llm
Settings.embed_model = embed_model
print("Settings.llm:", Settings.llm.model, "| Settings.embed_model:", Settings.embed_model.model_name)

Settings.llm: gpt-4o-mini | Settings.embed_model: text-embedding-3-small


### Checkpoint B

| You should see | Our run |
|---|---|
| Defaults | `OpenAI gpt-3.5-turbo`, `OpenAIEmbedding text-embedding-ada-002`, `chunk_size` 1024 |
| The vector | 1,536 numbers, largest difference from `embed()` 0.000000; 10 texts per request |
| `is_chat_model=False` | `is_chat_model: False`, `context_window: 3900`; a reply that starts with `**` and stops mid-sentence |
| `is_chat_model=True` | a one-sentence answer; `{'model': 'gpt-4o-mini', 'temperature': 0}` added to every request |
| `Settings` | `gpt-4o-mini` and `text-embedding-3-small` |

---

## Part C: Index the nodes and retrieve

Part C puts the 277 nodes in an index and searches one question with both pipelines.

A **`VectorStoreIndex`** embeds every node with `Settings.embed_model` and keeps the vectors in a vector store;
without one named, an in-memory `SimpleVectorStore`. Build it and look at what it holds:

In [11]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes)
print(type(index.vector_store).__name__, "|", len(index.docstore.docs), "nodes stored | first IDs:",
      list(index.docstore.docs)[:2])

SimpleVectorStore | 277 nodes stored | first IDs: ['S00-P00-C00', 'S00-P00-C01']


A `SimpleVectorStore` holding 277 nodes under their own IDs. The index keeps its own copy of each node, in `docstore`. A **retriever** searches the index for a
question and returns the nodes with their scores. Make one with `as_retriever()` and search the Tesla question:

In [12]:
retriever = index.as_retriever()
for hit in retriever.retrieve(question):
    print(f"  {hit.score:.4f}  {hit.node.node_id}")

  0.6664  S03-P00-C00
  0.6188  S03-P01-C00


Two rows, where Lab 01's `search()` returns three. As in Lab 05, the number comes from a default, and
the retriever's signature shows it:

In [13]:
import inspect

print(type(retriever).__name__, "|", inspect.signature(type(retriever)).parameters["similarity_top_k"])

VectorIndexRetriever | similarity_top_k: int = 2


LlamaIndex returns two nodes unless you pass `similarity_top_k`; LangChain's store returned four. Set it to 3 and search
the Tesla question both ways: your `search()` over the cached vectors, and the retriever. Predict first: the same chunks,
with the same scores?

In [14]:
from shared.search import chunk_vectors, search

vectors = chunk_vectors(corpus, settings)
retriever = index.as_retriever(similarity_top_k=3)
print("plain: search()")
for score, chunk in search(question, corpus, vectors, settings):
    print(f"  {score:.4f}  {chunk['chunk_id']}")
print("LlamaIndex: retriever.retrieve()")
for hit in retriever.retrieve(question):
    print(f"  {hit.score:.4f}  {hit.node.node_id}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

plain: search()


  0.7216  S03-P00-C00
  0.5404  S03-P01-C00
  0.4654  S03-P02-C00
LlamaIndex: retriever.retrieve()


  0.6664  S03-P00-C00
  0.6187  S03-P01-C00
  0.5884  S03-P02-C00

text-embedding-3-small at https://api.openai.com/v1


The same three chunks in the same order, but not the same scores: LlamaIndex gave 0.6664, 0.6187 and 0.5884 where `search()` gave 0.7216, 0.5404 and 0.4654. That is far more than the few thousandths two embeddings of the same text differ by, and the three scores sit much closer together. The two pipelines embedded something different. Part D asks how often that changes the order.

### Checkpoint C

| You should see | Our run |
|---|---|
| The index | `SimpleVectorStore | 277 nodes stored`, first IDs `S00-P00-C00`, `S00-P00-C01` |
| Default retriever | two rows; `similarity_top_k: int = 2` |
| `similarity_top_k=3` | the same three chunks as `search()`, but scores about 0.67, 0.62, 0.59 against 0.72, 0.54, 0.47 |

---

## Part D: Check the two searches on 30 questions

Part D is the parity check for the search: the same 30 questions and the same measures as Lab 05, in both pipelines.

First Lab 01's measure, known-source hit@3. A question is a hit when one of the nodes the retriever returns comes from
a paragraph in its `known_support`:

In [15]:
from shared.search import hit_at_k

plain_hits = hit_at_k(questions, corpus, vectors, settings)
li_results = [retriever.retrieve(q["question"]) for q in questions]
li_hits = sum(any(hit.node.metadata["source_id"] in q["known_support"] for hit in hits)
              for q, hits in zip(questions, li_results))
print(f"hit@3: plain {sum(plain_hits)} of {len(questions)} | LlamaIndex {li_hits} of {len(questions)}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

hit@3: plain 30 of 30 | LlamaIndex 30 of 30

text-embedding-3-small at https://api.openai.com/v1


30 and 30: both pipelines find a right paragraph for every question. Lab 05 taught you not to stop at a total. Compare what each pipeline returned, question by question:

In [16]:
plain_results = [search(q["question"], corpus, vectors, settings) for q in questions]
plain_ids = [[chunk["chunk_id"] for _, chunk in hits] for hits in plain_results]
li_ids = [[hit.node.node_id for hit in hits] for hits in li_results]
print(f"same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, li_ids))} of {len(questions)}")
differ = next(i for i, (a, b) in enumerate(zip(plain_ids, li_ids)) if a != b)
print(questions[differ]["question"])
print("  plain:     ", plain_ids[differ])
print("  LlamaIndex:", li_ids[differ])

same chunks in the same order: 15 of 30
Which NFL team represented the AFC at Super Bowl 50?
  plain:      ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00']
  LlamaIndex: ['S00-P00-C00', 'S00-P02-C00', 'S00-P01-C00']


15 of 30. On the first question that differs, the AFC question, both put `S00-P00-C00` first, but the other two differ: plain returns `S00-P01-C01` and `S00-P02-C00`, LlamaIndex `S00-P02-C00` and `S00-P01-C00`. This time `k` is the same, so the difference is in the vectors. Your `search()` embedded each
chunk's text. What did LlamaIndex embed? A node can show you the text each reader gets: `MetadataMode.EMBED` for the
embedding model, `MetadataMode.LLM` for the chat model. Print the first node as the embedding model read it:

In [17]:
from llama_index.core.schema import MetadataMode

print(nodes[0].get_content(metadata_mode=MetadataMode.EMBED))

chunk_id: S00-P00-C00
source_id: S00-P00
title: Super_Bowl_50
start: 0
end: 486
tokens: 98

Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the


Every metadata field sits on top of the text, one per line, and the vector was made from all of it. LlamaIndex does
this on purpose, since a file name or a title can help a search. Here it also embedded `start: 0`, `end: 486` and
`tokens: 98`, numbers that say nothing about the content but move every vector enough to swap close neighbours. Lab 03
added a title header on purpose and measured it; this one nobody chose.

`excluded_embed_metadata_keys` on each node lists the metadata names the embedding model must not see. Put every name
in it and print the first node again:

In [18]:
for node in nodes:
    node.excluded_embed_metadata_keys = list(node.metadata)
print(nodes[0].get_content(metadata_mode=MetadataMode.EMBED)[:120], "...")

Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 ...


Only the text is left. The index holds its own copies of the nodes, made before this change, so build it again, then
repeat both checks with the largest score difference between the pipelines:

In [19]:
index = VectorStoreIndex(nodes)
retriever = index.as_retriever(similarity_top_k=3)
li_results = [retriever.retrieve(q["question"]) for q in questions]
li_ids = [[hit.node.node_id for hit in hits] for hits in li_results]
li_hits = sum(any(hit.node.metadata["source_id"] in q["known_support"] for hit in hits)
              for q, hits in zip(questions, li_results))
gap = max(abs(plain_score - hit.score)
          for plain, hits in zip(plain_results, li_results)
          for (plain_score, _), hit in zip(plain, hits))
print(f"hit@3: LlamaIndex {li_hits} of {len(questions)}"
      f" | same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, li_ids))} of {len(questions)}")
print(f"largest score difference: {gap:.4f}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

hit@3: LlamaIndex 30 of 30 | same chunks in the same order: 30 of 30
largest score difference: 0.0008

text-embedding-3-small at https://api.openai.com/v1


Same chunks in the same order for 30 of 30 questions, hit@3 still 30 of 30, and the largest score difference 0.0008: the few thousandths Lab 01 measured between two embeddings of the same text. The search matches. As in Lab 05, the overall score agreed before the lists did, and the difference
was a default: there the number of chunks, here what the embedding model reads.

### Checkpoint D

| You should see | Our run |
|---|---|
| hit@3 | plain 30 of 30, LlamaIndex 30 of 30 |
| Same lists, metadata embedded | 15 of 30; the AFC question differs in its second and third chunks |
| What was embedded | the six metadata lines, then the text |
| Metadata excluded | hit@3 30 of 30, same lists 30 of 30, largest score difference a few thousandths (0.0008 in our run) |

---

## Part E: Ask through a query engine

Part E answers a question the way LlamaIndex's tutorials do, and prints what the engine sends to the model.

A **query engine** joins a retriever and a response step: it retrieves nodes, writes them into a prompt and asks
`Settings.llm`. `as_query_engine()` makes one from the index. Print what it holds:

In [20]:
query_engine = index.as_query_engine()
print(type(query_engine).__name__, "| retriever top k:", query_engine.retriever.similarity_top_k)
for name in query_engine.get_prompts():
    print(" ", name)

RetrieverQueryEngine | retriever top k: 2
  response_synthesizer:text_qa_template
  response_synthesizer:refine_template
  response_synthesizer:chat_content_qa_template
  response_synthesizer:chat_content_refine_template


The engine's retriever takes the default, 2, because no `similarity_top_k` was passed this time. Of its four
prompts, `text_qa_template` writes the question and the nodes; `refine_template` is used only when the nodes do not fit
in one call, and the two `chat_content_` prompts do the same for nodes that hold images or other media. Print the
messages `text_qa_template` writes for a chat model, with its two slots left as they are
([prompts](https://developers.llamaindex.ai/python/framework/module_guides/models/prompts/usage_pattern/)):

In [21]:
qa_prompt = query_engine.get_prompts()["response_synthesizer:text_qa_template"]
for message in qa_prompt.format_messages(llm=llm, context_str="{context_str}", query_str="{query_str}"):
    print(f"[{message.role.value}]\n{message.content}\n")

[system]
You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.

[user]
Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {query_str}
Answer: 



A system message from LlamaIndex, with rules of its own: answer from the context, and never refer to it. That second
rule works against citations, which point at the evidence on purpose. Ask the Tesla question:

In [22]:
response = query_engine.query(question)
print(type(response).__name__, "|", response.response)
for hit in response.source_nodes:
    print(f"  {hit.score:.4f}  {hit.node.node_id}")
print("\nThe first node as the model read it:")
print(response.source_nodes[0].node.get_content(metadata_mode=MetadataMode.LLM)[:200], "...")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

Response | Nikola Tesla was born on 10 July 1856.
  0.7221  S03-P00-C00
  0.5405  S03-P01-C00

The first node as the model read it:
chunk_id: S03-P00-C00
source_id: S03-P00
title: Nikola_Tesla
start: 0
end: 288
tokens: 66

Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) was a Serbian American inventor, ...

gpt-4o-mini at https://api.openai.com/v1


A `Response` with *Nikola Tesla was born on 10 July 1856.*, from two nodes, `S03-P00-C00` and `S03-P01-C00`. The answer is right, but it names no source; only `source_nodes` shows where it came from. The model read every metadata field too: `excluded_embed_metadata_keys` changed only what the
embedding model reads. The chat model has its own list, `excluded_llm_metadata_keys`, which Part F uses.

### Checkpoint E

| You should see | Our run |
|---|---|
| The engine | `RetrieverQueryEngine`, retriever top k 2, four prompts |
| The prompt | a system message that says *Never directly reference the given context in your answer.*, then the context and the query |
| Tesla | *Nikola Tesla was born on 10 July 1856.*, two source nodes; the first node read with its six metadata lines on top |

---

## Part F: Get a cited answer with `output_cls`

Part F rebuilds Lab 02's answer step in LlamaIndex: the same schema, the same rules, and a reply the quote check can
read.

Write the answer's shape as the Pydantic classes of Lab 05. A query engine takes one as `output_cls` and returns an
object of that class
([structured outputs](https://developers.llamaindex.ai/python/framework/module_guides/querying/structured_outputs/query_engine/)).
Try it with the chat model as it is:

In [23]:
from pydantic import BaseModel

class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str

class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]

cited_engine = index.as_query_engine(similarity_top_k=3, output_cls=CitedAnswer)
try:
    cited_engine.query(question)
except openai.BadRequestError as error:
    print(f"{type(error).__name__}: {error}")

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'tool_choice': 'tool_choice' is only allowed when 'tools' are specified.", 'type': 'invalid_request_error', 'param': 'tool_choice', 'code': None}}


A 400 from the API, shown on purpose. Unless told otherwise, LlamaIndex's OpenAI model asks for the answer as a tool
call and forces that call with `tool_choice`. But `OpenAILike` assumes the model has no tools (`is_function_calling_model=False`),
so LlamaIndex writes the schema into the prompt instead and still sends the `tool_choice`, with no tool.
`should_use_structured_outputs=True` sends the class as a strict JSON schema, Lab 02's `json_schema`.

Two more changes give the model Lab 02's prompt: `system_prompt` puts Lab 02's `RULES` in the system message, and
`text_qa_template` writes the user message as Lab 02 did. So that the model can name a chunk, it reads the chunk ID and
no other metadata (`excluded_llm_metadata_keys`). First the chat model and the nodes:

In [24]:
from llama_index.core import PromptTemplate
from shared import answer as answering

cited_llm = OpenAILike(model=settings["CHAT_MODEL"], api_base=chat_url(settings), api_key=os.environ["OPENAI_API_KEY"],
                       is_chat_model=True, additional_kwargs=options,
                       should_use_structured_outputs=True, system_prompt=answering.RULES)
for node in nodes:
    node.excluded_llm_metadata_keys = [name for name in node.metadata if name != "chunk_id"]
print(nodes[0].get_content(metadata_mode=MetadataMode.LLM)[:120], "...")

chunk_id: S00-P00-C00

Super Bowl 50 was an American football game to determine the champion of the National Football Le ...


The chat model now sends a strict schema and Lab 02's rules, and the model reads `chunk_id: S00-P00-C00`, then the
text. Next, the index is built again from these nodes, as in Part D, and the engine gets Lab 02's user message as its
`text_qa_template`:

In [25]:
index = VectorStoreIndex(nodes)
cited_engine = index.as_query_engine(llm=cited_llm, similarity_top_k=3, output_cls=CitedAnswer,
                                     text_qa_template=PromptTemplate("Question: {query_str}\nEvidence:\n{context_str}"))
print(type(cited_engine).__name__, "| retriever top k:", cited_engine.retriever.similarity_top_k)

RetrieverQueryEngine | retriever top k: 3


Ask the Tesla question, and the Moon question, which the corpus cannot answer. For the quote check, write the nodes the engine used as Lab 02's `[ID] text` lines:

In [26]:
for q in (question, "Who was the first person to walk on the Moon?"):
    response = cited_engine.query(q)
    reply = response.response
    lines = [f"[{hit.node.node_id}] {hit.node.text}" for hit in response.source_nodes]
    print(type(reply).__name__, "|", reply.answer)
    for claim in reply.claims:
        print(f"  quote found: {answering.quote_found(claim.model_dump(), lines)} | {claim.source_id} | {claim.quote}")

CitedAnswer | Nikola Tesla was born in the year 1856.
  quote found: True | S03-P00-C00 | 10 July 1856


CitedAnswer | Insufficient evidence.


The Tesla reply is a `CitedAnswer`, not text: *Nikola Tesla was born in the year 1856.*, with one claim from `S03-P00-C00` (*10 July 1856*) whose quote is found. The Moon question gets *Insufficient evidence.* and no claims, as Lab 02's abstention line asks.

### Checkpoint F

| You should see | Our run |
|---|---|
| `output_cls` alone | `BadRequestError: Error code: 400 ... Invalid value for 'tool_choice' ...`, printed in one line |
| What the model reads | `chunk_id: S00-P00-C00`, then the text |
| Tesla | *Nikola Tesla was born in the year 1856.*, `S03-P00-C00`, quote found: `True` |
| The Moon question | *Insufficient evidence.*, no claims |

---

## Part G: Compare the two pipelines on 30 questions

Part G finishes the parity check: Lab 02's `answer()` on your own search's three chunks, and the LlamaIndex engine, on
all 30 development questions, one answer each.

In [27]:
plain_replies, plain_lines, li_replies, li_lines = [], [], [], []
for q, hits in zip(questions, plain_results):
    lines = [f"[{chunk['chunk_id']}] {chunk['text']}" for _, chunk in hits]
    plain_lines.append(lines)
    plain_replies.append(answering.answer(q["question"], lines, settings))
    response = cited_engine.query(q["question"])
    li_lines.append([f"[{hit.node.node_id}] {hit.node.text}" for hit in response.source_nodes])
    li_replies.append(response.response.model_dump())
print(len(plain_replies), "answers from each pipeline | same evidence:",
      sum(a == b for a, b in zip(plain_lines, li_lines)), "of", len(questions))

30 answers from each pipeline | same evidence: 30 of 30


`model_dump()` turned each LlamaIndex answer into the dictionary Lab 02's `answer()` returns, so one function counts
both, with Lab 05's measures: refusals (these 30 questions should get none), claims and quotes found.

In [28]:
INSUFFICIENT = "Insufficient evidence."

def summary(replies, all_lines):
    refused = sum(reply["answer"].strip() == INSUFFICIENT for reply in replies)
    claims = [(lines, claim) for lines, reply in zip(all_lines, replies) for claim in reply["claims"]]
    found = sum(answering.quote_found(claim, lines) for lines, claim in claims)
    return f"refused {refused} of {len(replies)} | claims {len(claims)} | quotes found {found}"

def cited(reply):
    return [claim["source_id"] for claim in reply["claims"]]

print("plain:     ", summary(plain_replies, plain_lines))
print("LlamaIndex:", summary(li_replies, li_lines))
print("same cited IDs:", sum(cited(p) == cited(l) for p, l in zip(plain_replies, li_replies)), "of 30",
      "| same answer text:", sum(p["answer"] == l["answer"] for p, l in zip(plain_replies, li_replies)), "of 30")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

plain:      refused 0 of 30 | claims 30 | quotes found 30
LlamaIndex: refused 0 of 30 | claims 32 | quotes found 32
same cited IDs: 28 of 30 | same answer text: 24 of 30

gpt-4o-mini at https://api.openai.com/v1


Both refuse 0 of 30, and every quote is found: 30 claims plain, 32 LlamaIndex. The cited IDs agree for 28 of 30 questions and the answer text for 24 of 30. Print the answers whose text or cited IDs differ, or that have a quote the check did not find
(printed after the answer):

In [29]:
shown = 0
for plain, li, p_lines, l_lines in zip(plain_replies, li_replies, plain_lines, li_lines):
    missing = [c["quote"] for c in plain["claims"] if not answering.quote_found(c, p_lines)]
    missing += [c["quote"] for c in li["claims"] if not answering.quote_found(c, l_lines)]
    if plain["answer"] != li["answer"] or cited(plain) != cited(li) or missing:
        shown += 1
        print("plain:     ", cited(plain), plain["answer"])
        print("LlamaIndex:", cited(li), li["answer"], missing or "", "\n")
print(shown, "answers printed")

plain:      ['S00-P00-C00'] The Denver Broncos represented the AFC at Super Bowl 50.
LlamaIndex: ['S00-P00-C00'] The AFC was represented by the Denver Broncos at Super Bowl 50.  

plain:      ['S01-P01-C00'] There are over 100 different species of trees in the Saxon Garden.
LlamaIndex: ['S01-P01-C00'] Over 100 different species of trees can be found in the Saxon Garden.  

plain:      ['S02-P00-C00'] Normandy is located in France.
LlamaIndex: ['S02-P00-C00', 'S02-P01-C01'] Normandy is located in France.  

plain:      ['S02-P01-C02'] The duke in the Battle of Hastings was William the Conqueror.
LlamaIndex: ['S02-P01-C02'] William the Conqueror was the duke in the Battle of Hastings.  

plain:      ['S02-P02-C00'] The original meaning of the word Norman is "Northman" or "Norseman, Viking."
LlamaIndex: ['S02-P02-C00', 'S02-P00-C00'] The original meaning of the word Norman is "Norseman, Viking."  

plain:      ['S03-P00-C00'] Nikola Tesla was born on 10 July 1856.
LlamaIndex: ['S03-P00-C0

Five answers state the same fact from the same chunk in other words, such as *The AFC was represented by the Denver Broncos at Super Bowl 50.* against *The Denver Broncos represented the AFC at Super Bowl 50.* In the other two, LlamaIndex also cites a second chunk: *Normandy is located in France.*, with the same text, and the Norman answer, which also leaves out *Northman*. No quote failed the check. Lab 05 found that two runs of one pipeline give the same text for about 28 of 30 questions, so compare citations and quotes, not wording.

The parity table for the whole pipeline. The plain and LlamaIndex columns come from this run; the LangChain column is
Lab 05's saved run (`gpt-4o-mini`), since this environment does not install LangChain.

| Measure | Plain (Labs 01 and 02) | LangChain (Lab 05) | LlamaIndex |
|---|---|---|---|
| Known-source hit@3 on the 30 questions | 30 | 30 with `k=3` | 30 with `similarity_top_k=3` |
| Same chunks in the same order as plain | (the reference) | 30 of 30 with `k=3`; 0 of 30 at its default `k=4` | 30 of 30 with the metadata left out; 15 of 30 with it embedded |
| Refused, of 30 answerable questions | 0 | 0 | 0 |
| Claims, and quotes found | 30, 30 | 31, 31 | 32, 32 |
| Same cited IDs as plain | (the reference) | 29 of 30 | 28 of 30 |

And the defaults each framework chose for you, which this lab and Lab 05 had to find and set:

| What | LangChain (Lab 05) | LlamaIndex |
|---|---|---|
| Results per search | `k=4` | `similarity_top_k=2` |
| What the embedding model reads | the text | every metadata field, then the text |
| Models when you set none | `gpt-3.5-turbo` and `text-embedding-ada-002`, in a `ChatOpenAI()` or `OpenAIEmbeddings()` built without a model | the same two, also when you build no model at all: every index and engine takes them from `Settings` |
| Which API it calls | Chat Completions or Responses, chosen for you unless `use_responses_api=False` | Completions, unless `is_chat_model=True` |
| Prompt | yours | its own system message, which forbids referring to the context |
| Structured output | `with_structured_output` | `output_cls`, with `should_use_structured_outputs=True` on `OpenAILike` |

With every default set, both frameworks give your pipeline's results. What they change is how much you have to know
to get there: LlamaIndex does more for you in one call, and so hides more.

### Checkpoint G

| You should see | Our run |
|---|---|
| 30 questions | 30 answers from each pipeline; same evidence 30 of 30 |
| Measures | both refuse 0 of 30, 30 and 32 claims, every quote found; same cited IDs 28 of 30, same text 24 of 30 |
| Differences | 7 answers printed: five in other words from the same chunk, two with an extra citation; no quote that failed |

---

## What you can do now

- Turn chunk records into `TextNode`s that keep their IDs, and print what the embedding model and the chat model read
  from a node with `get_content(metadata_mode=...)`.
- Build LlamaIndex's chat model and embeddings from `models.env` with `OpenAILike` and `OpenAILikeEmbedding`, put them
  in `Settings`, and say what LlamaIndex uses when nothing is set and why `is_chat_model=True` matters.
- Index nodes in a `VectorStoreIndex`, search them with a retriever, and find its default `similarity_top_k` in the
  signature.
- Check a framework's search against your own question by question, and trace a difference to what was embedded.
- Read a query engine's prompts with `get_prompts()`, replace them, and get a cited answer with `output_cls` and
  `should_use_structured_outputs=True` that `quote_found()` checks.

## Challenge

Use only what this lab showed. Keep the printed results and your
answer to **Explain**.

1. **Choose what the embedding model reads.** Part D removed every metadata field from the embedded text, but a field
   can help: the policies' titles name their product (*Atlas reset*, *Boreal reset*). For *When does the password
   recovery email stop working for Atlas?*, Lab 01's search put the answering policy, H02, third, behind a Boreal policy.

   - Under **Task 1**, set `keep` to the metadata names the embedding model should read, starting with `["title"]`.
     The cell rebuilds the index and prints hit@1 for both pipelines, the same lists count and the Atlas question's top
     three.

   **Check:** hit@1, the same lists count and the Atlas question's top three. Our worked solution, with `["title"]`, got hit@1 30 of 30 (plain 29 of 30), the same lists as plain for 19 of 30 questions, and `['H01', 'H02', 'H04']` for the Atlas question, where plain search gives H01, H04, H02.

   **Explain:** did the title help, and where? Why is "same lists as plain" the wrong number for deciding what the
   embedding model reads?

**Task 1.** Choose the names to keep, then run the cell:

In [30]:
keep = None    # TODO: the metadata names the embedding model should read, such as ["title"]

if keep is None:
    print("⚠️ Choose the names to keep, then run the cell again.")
else:
    for node in nodes:
        node.excluded_embed_metadata_keys = [name for name in node.metadata if name not in keep]
    print(nodes[-1].get_content(metadata_mode=MetadataMode.EMBED), "\n")
    task_retriever = VectorStoreIndex(nodes).as_retriever(similarity_top_k=3)
    task_results = [task_retriever.retrieve(q["question"]) for q in questions]
    task_ids = [[hit.node.node_id for hit in hits] for hits in task_results]
    plain_first = sum(hits[0][1]["source_id"] in q["known_support"] for q, hits in zip(questions, plain_results))
    task_first = sum(hits[0].node.metadata["source_id"] in q["known_support"] for q, hits in zip(questions, task_results))
    print(f"hit@1: plain {plain_first} of 30 | LlamaIndex {task_first} of 30")
    print(f"same chunks in the same order as plain: {sum(a == b for a, b in zip(plain_ids, task_ids))} of 30")
    atlas = "When does the password recovery email stop working for Atlas?"
    print("Atlas question:", [hit.node.node_id for hit in task_retriever.retrieve(atlas)])
    print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

⚠️ Choose the names to keep, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'llama_index'` | the notebook runs on the repository's `.venv` kernel, which does not have LlamaIndex | Create this lab's environment (README) and choose the `Optional_LlamaIndex_Pipeline/.venv` kernel |
| `AuthenticationError: Error code: 401 ... Incorrect API key provided: fake.` | `OpenAILike` or `OpenAILikeEmbedding` built without `api_key`; their default key is the word `fake` | Pass `api_key=os.environ["OPENAI_API_KEY"]` after running Config |
| `TypeError: Completions.create() got an unexpected keyword argument 'reasoning_effort'` | `OpenAILike` without `is_chat_model=True` and `REASONING_EFFORT` set (for `gpt-6-luna`): the Completions call does not take that argument | Build the chat model with `is_chat_model=True`, as in Part B |
| `NotFoundError: Error code: 404 ... This is a chat model and not supported in the v1/completions endpoint.` | `OpenAILike` without `is_chat_model=True`, for a model that has no Completions endpoint | Build the chat model with `is_chat_model=True` |
| `BadRequestError: Error code: 400 ... Invalid value for 'tool_choice': 'tool_choice' is only allowed when 'tools' are specified.` | `output_cls` on a query engine whose `OpenAILike` has `should_use_structured_outputs=False`, the default | Build the chat model with `should_use_structured_outputs=True`, as in Part F |
| The retriever returns two nodes | `similarity_top_k` left at its default, 2 | Pass `similarity_top_k=3` to `as_retriever()` or `as_query_engine()` |

## Resources

- [LlamaIndex: documents and nodes](https://developers.llamaindex.ai/python/framework/module_guides/loading/documents_and_nodes/usage_documents/):
  metadata, and what the LLM and the embedding model read (`excluded_llm_metadata_keys`, `excluded_embed_metadata_keys`)
- [LlamaIndex: Settings](https://developers.llamaindex.ai/python/framework/module_guides/supporting_modules/settings/):
  the global defaults and how to set them
- [LlamaIndex reference: `OpenAILike`](https://developers.llamaindex.ai/python/framework-api-reference/llms/openai_like/)
  and [`OpenAILikeEmbedding`](https://developers.llamaindex.ai/python/framework-api-reference/embeddings/openai_like/):
  `is_chat_model`, `should_use_structured_outputs` and the other settings for OpenAI-compatible servers
- [LlamaIndex: prompts](https://developers.llamaindex.ai/python/framework/module_guides/models/prompts/usage_pattern/):
  `get_prompts()` and replacing a query engine's prompts
- [LlamaIndex: structured outputs from a query engine](https://developers.llamaindex.ai/python/framework/module_guides/querying/structured_outputs/query_engine/):
  `output_cls`

Back to [Lab 05](../05_Rebuild_With_LangChain/README.md), *How to rebuild a RAG pipeline with LangChain*, or to
[the lab list](../README.md).